# Hyperparameter Optimization

In [1]:
import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_curve, confusion_matrix, roc_auc_score
from catboost import CatBoostClassifier
from skopt.space import Integer, Real

import sys 
sys.path.append('..')  

from module.dataload import DPN_data
import ymlconfig

%matplotlib inline
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)  # decimal places for outputs from numpy
pd.set_option("display.precision", 3)  # decimal places for outputs from pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Load / Reload Selection Utility Functions

In [2]:
from utils2 import optimization as hpo

----

## Read Config File

In [3]:
config_path = Path(r'experiments\binary')
config_file = config_path / "optimization_config_dev.yml"
#config_file = config_path / "optimization_config_final.yml"
config_dict = ymlconfig.load_config(config_file)
config = ymlconfig.dict_to_namespace(config_dict)
config_dict

{'experiment': {'summary': 'binary classification - hyperparameter optimization(final experiment)',
  'classification_type': 'binary',
  'stage': 'hyperparameter_optimization',
  'tag': 'development',
  'verbosity': 1,
  'random_seed': 42},
 'data': {'dataset_path': '../dataset/Sudoscan Working File with Stats.xlsx'},
 'model': {'name': 'catboost'},
 'param_space': {'iterations': {'min': 100, 'max': 500},
  'depth': {'min': 4, 'max': 10},
  'learning_rate': {'min': 0.01, 'max': 0.1},
  'l2_leaf_reg': {'min': 1, 'max': 9}},
 'optimization': {'scoring': 'roc_auc',
  'k_splits_outer': 3,
  'n_repeats_outer': 2,
  'k_splits_inner': 3,
  'n_iter': 5},
 'evaluation': {'confidence': 0.95},
 'final_training': {'k_splits_inner': 3, 'n_iter': 5}}

## Data Loading

In [4]:
D = DPN_data(config.data.dataset_path)
D.load(classification=config.experiment.classification_type)

dfdpn = D.df
data_cols = dfdpn.drop(D.non_data_cols, axis=1, errors="ignore").columns
X = dfdpn[data_cols]
y = dfdpn['Confirmed_Binary_DPN']
X.shape, y.shape

((190, 40), (190,))

## Optuna Bayes Search Optimization

In [5]:
config.param_space

namespace(iterations=namespace(min=100, max=500),
          depth=namespace(min=4, max=10),
          learning_rate=namespace(min=0.01, max=0.1),
          l2_leaf_reg=namespace(min=1, max=9))

In [6]:
hpo.model_class[config.model.name]

catboost.core.CatBoostClassifier

In [7]:
config.optimization

namespace(scoring='roc_auc',
          k_splits_outer=3,
          n_repeats_outer=2,
          k_splits_inner=3,
          n_iter=5)

In [8]:
def param_space_fn(trial):
    return  {
        "iterations": trial.suggest_int(
            "iterations", 
            config.param_space.iterations.min, 
            config.param_space.iterations.max),
        "depth": trial.suggest_int(
            "depth", 
            config.param_space.depth.min, 
            config.param_space.depth.max),
        "learning_rate": trial.suggest_float(
            "learning_rate", 
            config.param_space.learning_rate.min, 
            config.param_space.learning_rate.max, 
            log=True),
        "l2_leaf_reg": trial.suggest_int(
            "l2_leaf_reg", 
            config.param_space.l2_leaf_reg.min, 
            config.param_space.l2_leaf_reg.max),
    }


In [9]:
opt_results = hpo.nested_cv_youden_optuna(
    X=X.values,
    y=y.values,
    model_class=hpo.model_class[config.model.name],   # class, not an instance
    param_space_fn=param_space_fn,
    n_splits_outer=config.optimization.k_splits_outer,
    n_repeats_outer=config.optimization.n_repeats_outer,
    n_splits_inner=config.optimization.k_splits_inner,
    n_iter=config.optimization.n_iter,   # Optuna trials per outer fold
    random_state=config.experiment.random_seed,
)

In [10]:
opt_results

{'roc_auc_mean': 0.9692212825933756,
 'roc_auc_std': 0.018776054590824542,
 'youden_mean': 0.7956483439041578,
 'youden_std': 0.08021401914153416,
 'sensitivity_mean': 0.9039816772374912,
 'specificity_mean': 0.8916666666666666,
 'threshold_mean': 0.6488591221384802,
 'threshold_std': 0.14354802298703395,
 'folds': [{'fold': 0,
   'roc_auc': 0.9840909090909091,
   'youden': 0.9090909090909092,
   'sensitivity': 0.9090909090909091,
   'specificity': 1.0,
   'threshold': 0.7137815889585889,
   'best_params': {'iterations': 433,
    'depth': 5,
    'learning_rate': 0.015199348301309814,
    'l2_leaf_reg': 2}},
  {'fold': 1,
   'roc_auc': 0.9755813953488373,
   'youden': 0.6499999999999999,
   'sensitivity': 1.0,
   'specificity': 0.65,
   'threshold': 0.3794176963815851,
   'best_params': {'iterations': 231,
    'depth': 10,
    'learning_rate': 0.04635431984752397,
    'l2_leaf_reg': 5}},
  {'fold': 2,
   'roc_auc': 0.9348837209302325,
   'youden': 0.8069767441860465,
   'sensitivity': 0

### Calculate Confidence Interval 

In [11]:
opt_ci  = hpo.mean_confidence_interval(opt_results, config)
opt_ci

youden 95.0% CI: {'mean': 0.7956483439041578, 'std': 0.08787005542393887, 'ci_lower': 0.7253389480563618, 'ci_upper': 0.8659577397519539, 'n_folds': 6}
roc_auc 95.0% CI: {'mean': 0.9692212825933756, 'std': 0.020568137280686065, 'ci_lower': 0.9527636475214215, 'ci_upper': 0.9856789176653297, 'n_folds': 6}


{'youden': {'mean': 0.7956483439041578,
  'std': 0.08787005542393887,
  'ci_lower': 0.7253389480563618,
  'ci_upper': 0.8659577397519539,
  'n_folds': 6},
 'roc_auc': {'mean': 0.9692212825933756,
  'std': 0.020568137280686065,
  'ci_lower': 0.9527636475214215,
  'ci_upper': 0.9856789176653297,
  'n_folds': 6}}

### Optimization Summary

In [12]:
opt_results_summary = {
    'youden': f'{opt_results["youden_mean"]:.3f} +/ {opt_results["youden_std"]:.3f}',
    'youden ci': f'{opt_ci["youden"]["ci_lower"]:.3f} - {opt_ci["youden"]["ci_upper"]:.3f}', 
    'roc_auc': f'{opt_results["roc_auc_mean"]:.3f} +/ {opt_results["roc_auc_std"]:.3f}',
    'roc_auc ci': f'{opt_ci["roc_auc"]["ci_lower"]:.3f} - {opt_ci["roc_auc"]["ci_upper"]:.3f}', 
    'threshold': f'{opt_results["threshold_mean"]:.3f} +/ {opt_results["threshold_std"]:.3f}',
    'specificity mean': f'{opt_results["specificity_mean"]:.3f}',
    'sensitivity mean': f'{opt_results["sensitivity_mean"]:.3f}',
}

opt_results_df = pd.DataFrame(opt_results_summary, index=['value']).T
opt_results_df

,value
youden,0.796 +/ 0.080
youden ci,0.725 - 0.866
roc_auc,0.969 +/ 0.019
roc_auc ci,0.953 - 0.986
threshold,0.649 +/ 0.144
specificity mean,0.892
sensitivity mean,0.904


 -----

### Train final model on ALL data

In [13]:
catboost_model = CatBoostClassifier(
    verbose=0,
    loss_function="Logloss",
    eval_metric="AUC",
    random_state=config.experiment.random_seed, 
    thread_count=-1
)

In [14]:
param_space = {
    'iterations': Integer(
        config.param_space.iterations.min, 
        config.param_space.iterations.max),
    'depth': Integer(
        config.param_space.depth.min, 
        config.param_space.depth.max),
    'learning_rate': Real(
        config.param_space.learning_rate.min, 
        config.param_space.learning_rate.max, 
        prior='log-uniform'),  # log-uniform better for LR
    'l2_leaf_reg': Real(
        config.param_space.l2_leaf_reg.min, 
        config.param_space.l2_leaf_reg.max, 
        prior='uniform'),
}

In [15]:
final_model, final_threshold, best_params = hpo.train_final_model(
    X=X.values, 
    y=y.values, 
    model=catboost_model, 
    param_space=param_space,
    n_splits_inner=config.final_training.k_splits_inner,
    n_iter=config.final_training.n_iter, 
    random_state=config.experiment.random_seed, 
    n_jobs=1
)

0:	learn: 0.6766401	total: 4.53ms	remaining: 1.76s
1:	learn: 0.6610091	total: 7.34ms	remaining: 1.43s
2:	learn: 0.6448277	total: 9.44ms	remaining: 1.22s
3:	learn: 0.6300781	total: 13.3ms	remaining: 1.29s
4:	learn: 0.6166119	total: 17ms	remaining: 1.31s
5:	learn: 0.6024873	total: 19.4ms	remaining: 1.25s
6:	learn: 0.5864141	total: 21.9ms	remaining: 1.2s
7:	learn: 0.5755168	total: 24ms	remaining: 1.15s
8:	learn: 0.5622360	total: 27ms	remaining: 1.15s
9:	learn: 0.5492061	total: 29.3ms	remaining: 1.12s
10:	learn: 0.5376998	total: 31.5ms	remaining: 1.09s
11:	learn: 0.5275073	total: 33.5ms	remaining: 1.06s
12:	learn: 0.5180197	total: 35.4ms	remaining: 1.03s
13:	learn: 0.5056234	total: 37.3ms	remaining: 1s
14:	learn: 0.4967230	total: 39ms	remaining: 978ms
15:	learn: 0.4862811	total: 41.5ms	remaining: 973ms
16:	learn: 0.4784654	total: 43.8ms	remaining: 964ms
17:	learn: 0.4701739	total: 46.1ms	remaining: 956ms
18:	learn: 0.4614774	total: 48.2ms	remaining: 943ms
19:	learn: 0.4521201	total: 50.5ms

In [16]:
print(best_params, final_threshold) 

OrderedDict({'depth': 6, 'iterations': 391, 'l2_leaf_reg': 8.462943990782671, 'learning_rate': 0.02069186296126172}) 0.671511316428229


### Final Model Sample Prediction

In [17]:
def test_final_model(model, threshold):
    test_size = 189
    Xnew = X.iloc[:test_size].values
    ypred, ypredproba = hpo.model_predict(Xnew, model, threshold)
    ynew=y.iloc[:test_size].values

    cm = confusion_matrix(ynew, ypred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    youden_test = (
        sensitivity + specificity - 1
        if not (np.isnan(sensitivity) or np.isnan(specificity))
        else np.nan
    )
    roc_auc = (
        roc_auc_score(ypred, ypredproba)
        if len(np.unique(ynew)) > 1
        else np.nan
    )

    print(youden_test, roc_auc)

test_final_model(final_model, final_threshold)

1.0 1.0


 -----

### Save artifacts and metrics

In [18]:
outputdir = config_path /  config.experiment.tag / config.experiment.stage
outputdir.mkdir(parents=True, exist_ok=True)
print(outputdir)

experiments\binary\development\hyperparameter_optimization


In [19]:
final_model_dict = {
    'name': config.model.name,
    'model': final_model,
    'best_params': best_params,
    'threshold' : final_threshold
}

In [20]:
joblib.dump(opt_results, outputdir / "optimization_results.joblib");
joblib.dump(final_model_dict, outputdir / "final_model_dict.joblib");
opt_results_df.to_csv(outputdir / "optimization_results_summary.csv")

### Load Save Results and Verify

In [21]:
loaded_optimization_results = joblib.load(outputdir / "optimization_results.joblib")
loaded_optimization_results

{'roc_auc_mean': 0.9692212825933756,
 'roc_auc_std': 0.018776054590824542,
 'youden_mean': 0.7956483439041578,
 'youden_std': 0.08021401914153416,
 'sensitivity_mean': 0.9039816772374912,
 'specificity_mean': 0.8916666666666666,
 'threshold_mean': 0.6488591221384802,
 'threshold_std': 0.14354802298703395,
 'folds': [{'fold': 0,
   'roc_auc': 0.9840909090909091,
   'youden': 0.9090909090909092,
   'sensitivity': 0.9090909090909091,
   'specificity': 1.0,
   'threshold': 0.7137815889585889,
   'best_params': {'iterations': 433,
    'depth': 5,
    'learning_rate': 0.015199348301309814,
    'l2_leaf_reg': 2}},
  {'fold': 1,
   'roc_auc': 0.9755813953488373,
   'youden': 0.6499999999999999,
   'sensitivity': 1.0,
   'specificity': 0.65,
   'threshold': 0.3794176963815851,
   'best_params': {'iterations': 231,
    'depth': 10,
    'learning_rate': 0.04635431984752397,
    'l2_leaf_reg': 5}},
  {'fold': 2,
   'roc_auc': 0.9348837209302325,
   'youden': 0.8069767441860465,
   'sensitivity': 0

In [22]:
loaded_final_model_dict = joblib.load(outputdir / "final_model_dict.joblib")
loaded_final_model_dict

{'name': 'catboost',
 'model': <catboost.core.CatBoostClassifier at 0x217676dc290>,
 'best_params': OrderedDict([('depth', 6),
              ('iterations', 391),
              ('l2_leaf_reg', 8.462943990782671),
              ('learning_rate', 0.02069186296126172)]),
 'threshold': 0.671511316428229}

In [23]:
test_final_model(
    loaded_final_model_dict["model"], 
    loaded_final_model_dict["threshold"]) 

1.0 1.0


In [24]:
pd.read_csv(outputdir / "optimization_results_summary.csv")

,Unnamed: 0,value
0,youden,0.796 +/ 0.080
1,youden ci,0.725 - 0.866
2,roc_auc,0.969 +/ 0.019
3,roc_auc ci,0.953 - 0.986
4,threshold,0.649 +/ 0.144
5,specificity mean,0.892
6,sensitivity mean,0.904
